# Healthcare Analytics Dashboard
Interactive KPI summary for Patient Flow, Care Quality & Outcomes, and Operational Efficiency.

In [ ]:
# Load gold tables
df_dept     = spark.read.format('delta').table('gold_department_daily_summary').limit(100000).toPandas()
df_scorecard = spark.read.format('delta').table('gold_department_scorecard').limit(100000).toPandas()
df_readmit  = spark.read.format('delta').table('gold_readmission_analysis').limit(100000).toPandas()
df_weekly   = spark.read.format('delta').table('gold_weekly_trends').limit(100000).toPandas()
df_flow     = spark.read.format('delta').table('gold_patient_flow').limit(100000).toPandas()

In [ ]:
total_admissions    = int(df_scorecard['total_admissions'].sum())
overall_readmit     = round(df_scorecard['total_readmissions'].sum() / total_admissions * 100, 1)
overall_avg_los     = round(df_dept['avg_los'].mean(), 1)
high_risk_pct       = round(df_scorecard['high_risk_count'].sum() / total_admissions * 100, 1)

print(f"Total Admissions     : {total_admissions:,}")
print(f"Readmission Rate     : {overall_readmit}%")
print(f"Avg Length of Stay   : {overall_avg_los} days")
print(f"High-Risk Patients   : {high_risk_pct}%")

In [ ]:
html = f"""
<!DOCTYPE html><html><head>
<meta charset="utf-8">
<title>Healthcare Analytics Dashboard</title>
<style>
  body {{ font-family: Segoe UI, Arial, sans-serif; background:#f4f6f9; margin:0; padding:20px; }}
  h1   {{ color:#0078d4; border-bottom:3px solid #0078d4; padding-bottom:8px; }}
  h2   {{ color:#323130; margin-top:30px; }}
  .kpis {{ display:flex; gap:16px; flex-wrap:wrap; margin:20px 0; }}
  .kpi  {{ background:#fff; border-radius:8px; padding:20px 28px; box-shadow:0 2px 6px rgba(0,0,0,.08);
           min-width:160px; text-align:center; }}
  .kpi .val {{ font-size:2em; font-weight:700; color:#0078d4; }}
  .kpi .lbl {{ font-size:.85em; color:#605e5c; margin-top:4px; }}
  table {{ border-collapse:collapse; width:100%; background:#fff; border-radius:8px;
           box-shadow:0 2px 6px rgba(0,0,0,.08); overflow:hidden; }}
  th    {{ background:#0078d4; color:#fff; padding:10px 14px; text-align:left; font-size:.9em; }}
  td    {{ padding:9px 14px; border-bottom:1px solid #edebe9; font-size:.88em; }}
  tr:hover td {{ background:#f3f9ff; }}
  .badge {{ display:inline-block; padding:2px 10px; border-radius:12px; font-size:.8em; font-weight:600; }}
  .Excellent {{ background:#dff6dd; color:#107c10; }}
  .Good      {{ background:#cceaff; color:#0078d4; }}
  .Needs.Improvement {{ background:#fff4ce; color:#b7610e; }}
  .Critical  {{ background:#fde7e9; color:#a80000; }}
</style></head><body>
<h1>🏥 Healthcare Analytics Dashboard</h1>

<div class="kpis">
  <div class="kpi"><div class="val">{total_admissions:,}</div><div class="lbl">Total Admissions</div></div>
  <div class="kpi"><div class="val">{overall_readmit}%</div><div class="lbl">Readmission Rate</div></div>
  <div class="kpi"><div class="val">{overall_avg_los}d</div><div class="lbl">Avg Length of Stay</div></div>
  <div class="kpi"><div class="val">{high_risk_pct}%</div><div class="lbl">High-Risk Patients</div></div>
</div>

<h2>Department Scorecard</h2>
<table>
  <tr><th>Department</th><th>Admissions</th><th>Readmission Rate</th><th>Avg LOS (days)</th><th>Daily Avg</th><th>Performance</th></tr>
  {''.join(f"""<tr>
    <td>{r['department']}</td>
    <td>{int(r['total_admissions']):,}</td>
    <td>{r['readmission_rate']}%</td>
    <td>{r['avg_los']}</td>
    <td>{r['daily_admissions_avg']}</td>
    <td><span class="badge {'Excellent' if r['readmission_rate']<=5 else 'Good' if r['readmission_rate']<=10 else 'Critical'}">
        {'Excellent' if r['readmission_rate']<=5 else 'Good' if r['readmission_rate']<=10 else 'Needs Improvement' if r['readmission_rate']<=15 else 'Critical'}
    </span></td></tr>""" for _, r in df_scorecard.sort_values('performance_rank').iterrows())}
</table>

<h2>Readmission Hotspots (rate &gt; 10%)</h2>
<table>
  <tr><th>Diagnosis Code</th><th>Insurance</th><th>Age Group</th><th>Admissions</th><th>Readmission Rate</th><th>Avg LOS</th></tr>
  {''.join(f"<tr><td>{r['primary_dx_code']}</td><td>{r['insurance_type']}</td><td>{r['age_group']}</td><td>{int(r['total_admissions'])}</td><td>{r['readmission_rate']}%</td><td>{round(r['avg_los'],1)}</td></tr>"
    for _, r in df_readmit[df_readmit['readmission_rate']>10].sort_values('readmission_rate', ascending=False).head(20).iterrows())}
</table>
</body></html>
"""

displayHTML(html)